In [ ]:
import json
with open('../Data_Files/wanted_detail_improve_20250604.json','r', encoding='utf-8') as f:
    job_postings = json.load(f)

for job_postings in job_postings.values():
    title = job_postings.get("제목","무제")
    intro = job_postings.get("회사 소개",''),
    main_task = job_postings.get("주요 업무", []),
    requirements = job_postings.get("자격 요건", []),
    preferred_points = job_postings.get("우대 사항", []),
    benefits = job_postings.get("복지", ''),
    hire_rounds = job_postings.get("채용절차", ''),
    attraction = job_postings.get("장점", ''),
    company = job_postings.get("회사명", ''),
    location = job_postings.get("근무지", ''),
    district = job_postings.get("지역", ''),
    category_parent = job_postings.get("직군",''),
    category_child = job_postings.get("직무", ''),
    skill = job_postings.get("기술 스택", []),
    annual_from = job_postings.get("요구 최소 경력", ""),
    annual_to = job_postings.get("요구 최대 경력", ""),
    is_newbie = job_postings.get("신입 지원 가능 여부", ""),
    due_time = job_postings.get("마감일", "상시 공고"),

    print(title)


In [ ]:
import pandas as pd
df = pd.read_json('../Data_Files/wanted_detail_improve_20250604.json', orient='index')


df


In [ ]:
job_postings = df.to_dict(orient='records')

job_postings[0].get("제목","무제")

In [ ]:
import json
import os
from Run_Pipeline.Env_Loader import EnvLoader
import openai

# 1) OpenAI API 키 불러오기 (환경 변수)
EnvLoader.load_local('../.env')

'''
채용공고 json 파일을 불러와서 , 해당 파일을 기준으로 프롬프트 설정을해서 QA 세트를 생성한다
그리고, 생성된 QA 세트를 Gemma3 학습 포맷에 맞춰 input/target 형태로 변환하여 JSONL 파일로 저장한다.

채용공고 JSON 파일은 다음과 같은 구조를 가집니다:
{
    "제목": "채용공고 제목",
    "회사 소개": "회사에 대한 소개",
    "주요 업무": ["업무1", "업무2", ...],
    "우대 사항": ["우대사항1", "우대사항2", ...]
}

QA세트는 너무 일반적이지 않고, 채용공고의 특성을 반영하여 생성됩니다.

'''

client = openai.OpenAI()


def generate_qa_pairs_for_posting(job_postings: dict) -> list:
    """
    GPT-4o-mini로 QA 쌍을 생성하는 부분은 이전 예시와 동일합니다.
    반환값: [{"question": "...", "answer": "..."}, ...]
    """
    title = job_postings.get("제목","무제")
    intro = job_postings.get("회사 소개",''),
    main_task = job_postings.get("주요 업무", []),
    requirements = job_postings.get("자격 요건", []),
    preferred_points = job_postings.get("우대 사항", []),
    benefits = job_postings.get("복지", ''),
    hire_rounds = job_postings.get("채용절차", ''),
    attraction = job_postings.get("장점", ''),
    company = job_postings.get("회사명", ''),
    location = job_postings.get("근무지", ''),
    district = job_postings.get("지역", ''),
    category_parent = job_postings.get("직군",''),
    category_child = job_postings.get("직무", ''),
    skill = job_postings.get("기술 스택", []),
    annual_from = job_postings.get("요구 최소 경력", ""),
    annual_to = job_postings.get("요구 최대 경력", ""),
    is_newbie = job_postings.get("신입 지원 가능 여부", ""),
    due_time = job_postings.get("마감일", "상시 공고"),

    prompt = f"""
아래는 하나의 채용공고 정보입니다.
'직무', '주요 업무', '우대사항', '회사 소개','기술스택' 부분만 이용해서
질문(Question)과 답변(Answer) 쌍을 JSON 배열 형식으로 생성해 주세요.
질문은 실제 신입/주니어 레벨의 사용자가 주로 질문하는 형태로 질문을 작성합니다.
답변(A)은 채용공고 정보를 참고하되, 연봉·마감일 등 가변적 수치는 생략하고, 해당 기업의 실무자나 인사담당자가 전문적으로 답변하는 형태로 작성합니다.
출력 예시:
[
  {{
    "question": "질문 텍스트",
    "answer": "답변 텍스트"
  }},
  ...
]

---
채용공고 정보:
제목: {title}

회사 소개:
{intro}

주요 업무:
{main_task}

우대사항:
{preferred_points}

자격 요건:
{requirements}

복지:
{benefits}

기술 스택:
{skill}

신입 지원 가능 여부:
{is_newbie}

요구 최소 경력:
{annual_from}

---

위 내용을 바탕으로 4~6개의 일관된 QA 쌍을 만들어 주세요.
JSON 배열 형태로, 공백이나 주석 없이 순수한 JSON만 반환해 주세요.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 채용 공고를 바탕으로 QA 쌍을 생성하는 전문가입니다."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.0,
        max_tokens=800,
        top_p=1.0,
    )
    generated = response.choices[0].message.content.strip()

    try:
        qa_list = json.loads(generated)
    except json.JSONDecodeError:
        start = generated.find("[")
        end   = generated.rfind("]") + 1
        qa_list = json.loads(generated[start:end])

    return qa_list

In [ ]:
qa_list = generate_qa_pairs_for_posting(job_postings[0])

In [ ]:
qa_list

In [ ]:
def wrap_for_gemma3(question: str, answer: str) -> dict:
    """
    Gemma3 학습 포맷에 맞춰 input/target 을 생성해 줍니다.
    """
    # 1) input 문자열 생성
    input_text = (
        "<bos>\n"
        "<start_of_turn>user\n"
        "시스템: 너는 채용 공고 정보를 기반으로 QA를 수행하는 챗봇입니다.\n"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"질문: {question}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model"
    )

    # 2) target 문자열 생성
    target_text = f"{answer}\n<end_of_turn><eos>"

    return {"input": input_text, "target": target_text}

In [ ]:
wr = wrap_for_gemma3(qa_list[0].get("question", ""), qa_list[0].get("answer", ""))
wr

In [ ]:
if __name__ == "__main__":
    # 3) 로컬 채용공고 JSON 불러오기 (리스트 형태)
    with open("job_postings.json", "r", encoding="utf-8") as f:
        job_postings = json.load(f)

    # 4) 전체 결과 저장용 리스트
    gemma3_data = []

    for posting in job_postings:
        title = posting.get("제목", "무제 공고")
        print(f"▶ QA 생성 중: {title}")
        qa_pairs = generate_qa_pairs_for_posting(posting)

        # 5) 각 QA 쌍마다 Gemma3 input/target 생성
        for qa in qa_pairs:
            q = qa.get("question", "").strip()
            a = qa.get("answer", "").strip()
            if not q or not a:
                continue
            wrapped = wrap_for_gemma3(q, a)
            gemma3_data.append(wrapped)

    # 6) JSONL 포맷으로 저장
    out_path = "job_postings_gemma3.jsonl"
    with open(out_path, "w", encoding="utf-8") as fout:
        for entry in gemma3_data:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\n✅ Gemma3 형식 JSONL 생성 완료 → {out_path}")

# 다른 방식 아예 QA 페어만 생성하는 코드

In [91]:
def generate_gemma3_peft_qa_strict_json(num_pairs: int, output_path: str = "gemma3_peft_qa.jsonl"):
    """
    Gemma3:4b PEFT 학습용으로, num_pairs 만큼의 QA(input/output) 페어를
    모델이 알아서 다양한 직군을 골라 생성하도록 요청한 뒤,
    *반드시* JSON 배열만 출력하도록 강제하고,
    JSON Lines(.jsonl) 형식으로 output_path에 저장합니다.

    * num_pairs: 생성할 QA 페어 개수 (처음엔 5~10 정도로 테스트 권장)
    * output_path: 저장 파일 경로
    """
    # ─── 2. Prompt 작성 ───
    # - "응답은 오직 JSON 배열 하나만 달라"는 부분을 최대한 강조
    # - 따옴표, 중괄호 등을 정확히 맞춰 달라고 지시
    prompt = f"""
다음 요구사항을 정확히 지켜서 **오로지 JSON 배열** 하나만 출력하세요.
절대로 다른 설명, 번호 매기기, 마크다운, 코드블록 등을 포함하지 마시고,
배열의 시작({{)과 끝(}})만 반환하십시오.

요구사항:
1. 총 {num_pairs}개의 QA 페어를 생성합니다.
2. 각 요소는 **딱 두 개의 필드**만 가집니다:
   - "input": 취업 준비생이 실제로 할 법한 질문 텍스트
   - "output": 각 직군의 시니어(선배)가 실무 경험을 바탕으로 상세하게 조언한 답변 텍스트
3. 모델이 스스로 다양한 직군(software engineer, data scientist, marketing manager, UX 디자이너, 금융 애널리스트 등)을 떠올려서, 균형 있게 분배해 주세요.
4. 최종 출력 예시(반드시 중괄호를 이스케이핑 없이 아래 형식 그대로):
[
  {{
    "input": "OO회사 소프트웨어 엔지니어 포지션에 지원하려면 어떤 기술을 보강해야 하나요?"",
    "output": "시니어 입장에서 ... (여기에 상세한 조언)"
  }},
  ...
]

5. JSON 배열의 요소는 쉼표(,)로 구분하며, **총 {num_pairs}개**여야 합니다.
6. 출력 시 절대 빈 줄(whitespace)도 포함하지 말고, **맨 처음 문자**가 `[`(대괄호 시작)이고, **맨 마지막 문자**가 `]`(대괄호 끝)이어야 합니다.
"""

    # ─── 3. ChatCompletion 호출 (openai>=1.x) ───
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 다양한 산업 분야의 시니어 전문가들이 모여 있는 취업 컨설팅 위원회입니다."},
            {"role": "user",   "content": prompt.strip()}
        ],
        temperature=0.7,
        # num_pairs=10일 때는 max_tokens=2000 정도로도 충분할 수 있음
        max_tokens=2000
    )

    # ─── 4. 결과 텍스트 추출 ───
    text = response.choices[0].message.content.strip()

    # ─── 5. Raw response 확인 ───
    print("\n===== Raw response from model =====")
    print(text)
    print("===== End of raw response =====\n")

    # ─── 6. JSON 형식 검증 ───
    if not text.startswith("[") or not text.endswith("]"):
        # JSON 배열 형식이 아닐 경우
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)
        raise RuntimeError(
            "모델이 JSON 배열로 정확히 응답하지 않았습니다.\n"
            f"원본 텍스트를 '{output_path}'에 저장했습니다. "
            "Raw response를 확인하고 prompt를 조정하세요."
        )

    # ─── 7. JSON 파싱 시도 ───
    try:
        qa_list = json.loads(text)
    except json.JSONDecodeError as e:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)
        raise RuntimeError(
            f"파싱 에러가 발생했습니다: {e}\n원본 텍스트를 '{output_path}'에 저장했습니다.\n"
            "Raw response를 확인하고 JSON 포맷을 맞춰주세요."
        )

    # ─── 8. JSON Lines 형식으로 저장 ───
    with open(output_path, "w", encoding="utf-8") as f:
        for obj in qa_list:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    print(f"✅ 생성된 {len(qa_list)}개의 Gemma3:4b PEFT용 QA 페어를 '{output_path}'에 저장했습니다.")
    return qa_list


In [92]:
NUM_PAIRS = 10
OUTPUT_FILE = "gemma3_peft_qa.jsonl"

qa_pairs = generate_gemma3_peft_qa_strict_json(NUM_PAIRS, output_path=OUTPUT_FILE)

# 생성된 QA 중 앞 3개만 출력해서 확인
for i, qa in enumerate(qa_pairs[:3], start=1):
    print(f"\n==== QA #{i} ====")
    print(f"INPUT  : {qa.get('input')}")
    print(f"OUTPUT : {qa.get('output')[:200]}...")  # 답변 앞부분 일부만


===== Raw response from model =====
[
  {
    "input": "소프트웨어 엔지니어 포지션에 지원하려면 어떤 기술을 보강해야 하나요?",
    "output": "시니어 입장에서 Java, Python, 그리고 데이터베이스 기술이 중요합니다. 또한, 알고리즘과 자료구조에 대한 이해를 깊게 하는 것이 좋습니다."
  },
  {
    "input": "데이터 사이언티스트로 진로를 바꾸고 싶습니다. 어떤 스킬을 먼저 배워야 할까요?",
    "output": "R과 Python을 배우는 것이 필수적입니다. 데이터 분석과 시각화 도구인 Tableau도 익히는 것이 좋습니다."
  },
  {
    "input": "마케팅 매니저로서 경쟁사 분석을 어떻게 수행해야 하나요?",
    "output": "경쟁사의 마케팅 전략을 연구하고, 그들의 소셜 미디어 활동을 분석하는 것이 중요합니다. SWOT 분석을 활용하세요."
  },
  {
    "input": "UX 디자이너로서 포트폴리오에 포함해야 할 필수 요소는 무엇인가요?",
    "output": "사용자 연구 결과, 와이어프레임, 프로토타입을 포함해야 합니다. 또한, 실제 사용자 피드백을 반영한 사례를 강조하세요."
  },
  {
    "input": "금융 애널리스트로서 어떤 자격증을 취득하는 것이 유리한가요?",
    "output": "CFA 자격증이 특히 유용합니다. 또한, 금융 모델링 및 기업 가치 평가에 대한 전문 지식을 쌓는 것도 중요합니다."
  },
  {
    "input": "소프트웨어 엔지니어로서 협업을 잘하기 위한 팁은 무엇인가요?",
    "output": "명확한 커뮤니케이션과 코드 리뷰 프로세스를 통해 팀원들과의 협업을 강화하세요. Agile 방법론을 이해하는 것도 도움이 됩니다."
  },
  {
    "input": "데이터 사이언티스트가 되기 위해 필요한 경험은 어떤 것인가요?",
    "output": "실제 데이터 